# Phase 1: ORB Detection Validation

**Goal:** Match Pine script ORB logic exactly

## Pine Logic (Source of Truth)

```pine
// Session detection
sessionTime = "0930-0935"  // 5-minute ORB
inSession = not na(time(timeframe.period, sessionTime, "America/New_York"))
isNewSession = inSession and not inSession[1]

// ORB High/Low tracking
if isNewSession
    sessionHigh := high
    sessionLow := low

if inSession and not isNewSession
    sessionHigh := math.max(sessionHigh, high)
    sessionLow := math.min(sessionLow, low)

// ORB Complete
orbComplete = not inSession and not na(sessionHigh) and not na(sessionLow)

// Breakout conditions
baseBreakoutHigh = orbComplete and close > sessionHigh
baseBreakoutLow = orbComplete and close < sessionLow

breakoutThreshold = atr * 0.1  // breakoutThresholdMultiplier
sufficientBreakoutHigh = close > sessionHigh + breakoutThreshold
sufficientBreakoutLow = close < sessionLow - breakoutThreshold

candleBody = math.abs(close - open)
candleRange = high - low
bodyStrengthRatio = candleRange > 0 ? candleBody / candleRange : 0
strongCandleBody = bodyStrengthRatio >= 0.5  // minBodyStrengthRatio
```

In [12]:
# Cell 1: Setup
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
import os

# Set API key
os.environ["POLYGON_API_KEY"] = "RQ5xXMXAaktwTevFdi26WlsBC6XG5qUO"  # <-- PUT YOUR KEY

from data_collector import PolygonDataCollector

print("✓ Setup complete")

✓ Setup complete


In [13]:
# Cell 2: Fetch data
collector = PolygonDataCollector()
df = collector.fetch_bars('NVDA', days_back=60, bar_size=1)
print(f"✓ Loaded {len(df)} bars")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 60 days)...
  ✓ Loading from cache (0.3 hours old)
✓ Loaded 16064 bars


In [14]:
# Cell 3: ORB Detection - MATCHING PINE EXACTLY

def detect_orb_session(df, orb_start_hour=9, orb_start_minute=30, orb_minutes=5):
    """
    Pine Logic:
        sessionTime = "0930-0935"
        inSession = not na(time(timeframe.period, sessionTime, "America/New_York"))
        isNewSession = inSession and not inSession[1]
    """
    # Calculate ORB end time
    orb_end_minute = orb_start_minute + orb_minutes
    orb_end_hour = orb_start_hour
    if orb_end_minute >= 60:
        orb_end_hour += 1
        orb_end_minute -= 60
    
    # inSession: True if bar is within ORB window
    # Pine: time(timeframe.period, "0930-0935", "America/New_York")
    # This returns non-na if the bar's timestamp falls within the session
    
    def is_in_orb_session(ts):
        bar_hour = ts.hour
        bar_minute = ts.minute
        bar_total = bar_hour * 60 + bar_minute
        
        orb_start_total = orb_start_hour * 60 + orb_start_minute
        orb_end_total = orb_end_hour * 60 + orb_end_minute
        
        # Bar at 09:30 is IN session
        # Bar at 09:35 is OUT of session (ORB complete)
        return orb_start_total <= bar_total < orb_end_total
    
    df = df.copy()
    df['in_session'] = df.index.map(is_in_orb_session)
    df['is_new_session'] = df['in_session'] & ~df['in_session'].shift(1, fill_value=False)
    
    return df


def calculate_orb_levels(df):
    """
    Pine Logic:
        if isNewSession
            sessionHigh := high
            sessionLow := low
        if inSession and not isNewSession
            sessionHigh := math.max(sessionHigh, high)
            sessionLow := math.min(sessionLow, low)
    """
    df = df.copy()
    df['orb_high'] = np.nan
    df['orb_low'] = np.nan
    
    current_high = np.nan
    current_low = np.nan
    
    for i in range(len(df)):
        if df['is_new_session'].iloc[i]:
            # New session starts
            current_high = df['high'].iloc[i]
            current_low = df['low'].iloc[i]
        elif df['in_session'].iloc[i]:
            # Still in session, update high/low
            current_high = max(current_high, df['high'].iloc[i])
            current_low = min(current_low, df['low'].iloc[i])
        
        # Record current ORB levels
        df.iloc[i, df.columns.get_loc('orb_high')] = current_high
        df.iloc[i, df.columns.get_loc('orb_low')] = current_low
    
    return df


def mark_orb_complete(df):
    """
    Pine Logic:
        orbComplete = not inSession and not na(sessionHigh) and not na(sessionLow)
    """
    df = df.copy()
    df['orb_complete'] = ~df['in_session'] & df['orb_high'].notna() & df['orb_low'].notna()
    return df


# Apply the logic
df = detect_orb_session(df)
df = calculate_orb_levels(df)
df = mark_orb_complete(df)

print("✓ ORB detection complete")
print(f"\nSession bars: {df['in_session'].sum()}")
print(f"New sessions: {df['is_new_session'].sum()}")
print(f"ORB complete bars: {df['orb_complete'].sum()}")

✓ ORB detection complete

Session bars: 210
New sessions: 42
ORB complete bars: 15854


In [15]:
# Cell 4: Pick a specific day to validate
# Let's look at 2025-10-29 (the first trade we compared)

test_date = '2025-10-29'
day_df = df[df.index.date.astype(str) == test_date].copy()

print(f"Bars on {test_date}: {len(day_df)}")
print(f"\nORB Session bars (9:30-9:35):")
orb_bars = day_df[day_df['in_session']]
print(orb_bars[['open', 'high', 'low', 'close', 'in_session', 'is_new_session']])

Bars on 2025-10-29: 391

ORB Session bars (9:30-9:35):
                               open    high       low     close  in_session  \
timestamp                                                                     
2025-10-29 09:30:00-04:00  207.9800  208.07  207.0900  207.6950        True   
2025-10-29 09:31:00-04:00  207.6700  209.40  207.6500  209.3232        True   
2025-10-29 09:32:00-04:00  209.3050  209.89  209.0600  209.6022        True   
2025-10-29 09:33:00-04:00  209.6099  210.39  209.5253  210.1800        True   
2025-10-29 09:34:00-04:00  210.1900  211.08  209.9200  210.7900        True   

                           is_new_session  
timestamp                                  
2025-10-29 09:30:00-04:00            True  
2025-10-29 09:31:00-04:00           False  
2025-10-29 09:32:00-04:00           False  
2025-10-29 09:33:00-04:00           False  
2025-10-29 09:34:00-04:00           False  


In [16]:
# Cell 5: Show ORB levels for that day
print(f"\nORB Levels for {test_date}:")
print(f"  ORB High: {day_df['orb_high'].iloc[-1]:.2f}")
print(f"  ORB Low:  {day_df['orb_low'].iloc[-1]:.2f}")
print(f"  ORB Range: ${day_df['orb_high'].iloc[-1] - day_df['orb_low'].iloc[-1]:.2f}")

# Show first post-ORB bars
print(f"\nFirst bars after ORB complete:")
post_orb = day_df[day_df['orb_complete']].head(10)
print(post_orb[['open', 'high', 'low', 'close', 'orb_high', 'orb_low', 'orb_complete']])


ORB Levels for 2025-10-29:
  ORB High: 211.08
  ORB Low:  207.09
  ORB Range: $3.99

First bars after ORB complete:
                               open      high       low     close  orb_high  \
timestamp                                                                     
2025-10-29 09:35:00-04:00  210.8000  211.4700  210.7648  210.9450    211.08   
2025-10-29 09:36:00-04:00  210.9400  211.6300  210.8500  211.3250    211.08   
2025-10-29 09:37:00-04:00  211.3360  211.3790  210.0900  210.1800    211.08   
2025-10-29 09:38:00-04:00  210.1750  210.6901  210.0200  210.1950    211.08   
2025-10-29 09:39:00-04:00  210.1950  210.6200  210.0100  210.0200    211.08   
2025-10-29 09:40:00-04:00  210.0200  210.1100  209.6800  209.8500    211.08   
2025-10-29 09:41:00-04:00  209.8800  210.0000  209.6400  209.8600    211.08   
2025-10-29 09:42:00-04:00  209.8700  210.6500  209.7700  210.5125    211.08   
2025-10-29 09:43:00-04:00  210.5200  211.0700  210.3000  210.9900    211.08   
2025-10-29 09:

In [17]:
# Cell 6: ATR Calculation - matching Pine

def calc_atr(df, period=14):
    """
    Pine: ta.atr(14)
    True Range = max(high-low, abs(high-prev_close), abs(low-prev_close))
    ATR = RMA of True Range (Wilder's smoothing)
    """
    df = df.copy()
    
    # True Range
    tr1 = df['high'] - df['low']
    tr2 = abs(df['high'] - df['close'].shift(1))
    tr3 = abs(df['low'] - df['close'].shift(1))
    df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    
    # RMA (Wilder's smoothing) - Pine's ta.atr uses this
    # RMA = alpha * current + (1-alpha) * previous, where alpha = 1/period
    alpha = 1.0 / period
    df['atr'] = df['tr'].ewm(alpha=alpha, adjust=False).mean()
    
    return df

df = calc_atr(df, period=14)
print("✓ ATR calculated")

# Show ATR for test date
day_df = df[df.index.date.astype(str) == test_date]
print(f"\nATR at 9:35 on {test_date}: {day_df['atr'].iloc[5]:.4f}")

✓ ATR calculated

ATR at 9:35 on 2025-10-29: 0.9064


In [18]:
# Cell 7: Breakout Detection - matching Pine EXACTLY

def detect_breakouts(df, breakout_threshold_mult=0.1, min_body_strength=0.5):
    """
    Pine Logic:
        baseBreakoutHigh = orbComplete and close > sessionHigh
        baseBreakoutLow = orbComplete and close < sessionLow
        
        breakoutThreshold = atr * breakoutThresholdMultiplier
        sufficientBreakoutHigh = close > sessionHigh + breakoutThreshold
        sufficientBreakoutLow = close < sessionLow - breakoutThreshold
        
        candleBody = math.abs(close - open)
        candleRange = high - low
        bodyStrengthRatio = candleRange > 0 ? candleBody / candleRange : 0
        strongCandleBody = bodyStrengthRatio >= minBodyStrengthRatio
    """
    df = df.copy()
    
    # Base breakout (price crosses ORB level)
    df['base_breakout_high'] = df['orb_complete'] & (df['close'] > df['orb_high'])
    df['base_breakout_low'] = df['orb_complete'] & (df['close'] < df['orb_low'])
    
    # Sufficient breakout (exceeds threshold)
    df['breakout_threshold'] = df['atr'] * breakout_threshold_mult
    df['sufficient_high'] = df['close'] > (df['orb_high'] + df['breakout_threshold'])
    df['sufficient_low'] = df['close'] < (df['orb_low'] - df['breakout_threshold'])
    
    # Body strength
    df['candle_body'] = abs(df['close'] - df['open'])
    df['candle_range'] = df['high'] - df['low']
    df['body_ratio'] = df['candle_body'] / df['candle_range'].replace(0, np.nan)
    df['strong_body'] = df['body_ratio'] >= min_body_strength
    
    # Combined breakout signal
    df['breakout_high'] = (
        df['base_breakout_high'] & 
        df['sufficient_high'] & 
        df['strong_body']
    )
    df['breakout_low'] = (
        df['base_breakout_low'] & 
        df['sufficient_low'] & 
        df['strong_body']
    )
    
    return df


df = detect_breakouts(df)
print("✓ Breakout detection complete")
print(f"\nTotal breakout_high signals: {df['breakout_high'].sum()}")
print(f"Total breakout_low signals: {df['breakout_low'].sum()}")

✓ Breakout detection complete

Total breakout_high signals: 2111
Total breakout_low signals: 2773


In [19]:
# Cell 8: Find breakouts on our test date

day_df = df[df.index.date.astype(str) == test_date].copy()

print(f"\n=== Breakout Analysis for {test_date} ===")
print(f"\nORB High: {day_df['orb_high'].iloc[-1]:.2f}")
print(f"ORB Low:  {day_df['orb_low'].iloc[-1]:.2f}")

# Find first breakout
breakouts_high = day_df[day_df['breakout_high']]
breakouts_low = day_df[day_df['breakout_low']]

print(f"\nHIGH breakouts: {len(breakouts_high)}")
if len(breakouts_high) > 0:
    first_high = breakouts_high.iloc[0]
    print(f"  First at: {breakouts_high.index[0]}")
    print(f"  Close: {first_high['close']:.2f}")
    print(f"  ORB High + Threshold: {first_high['orb_high'] + first_high['breakout_threshold']:.2f}")
    print(f"  Body ratio: {first_high['body_ratio']:.2f}")

print(f"\nLOW breakouts: {len(breakouts_low)}")
if len(breakouts_low) > 0:
    first_low = breakouts_low.iloc[0]
    print(f"  First at: {breakouts_low.index[0]}")
    print(f"  Close: {first_low['close']:.2f}")
    print(f"  ORB Low - Threshold: {first_low['orb_low'] - first_low['breakout_threshold']:.2f}")
    print(f"  Body ratio: {first_low['body_ratio']:.2f}")


=== Breakout Analysis for 2025-10-29 ===

ORB High: 211.08
ORB Low:  207.09

HIGH breakouts: 3
  First at: 2025-10-29 09:59:00-04:00
  Close: 211.97
  ORB High + Threshold: 211.15
  Body ratio: 0.94

LOW breakouts: 74
  First at: 2025-10-29 11:47:00-04:00
  Close: 206.62
  ORB Low - Threshold: 207.05
  Body ratio: 0.98


In [20]:
# Cell 9: Compare with TradingView

print("""\n=== VALIDATION CHECKLIST ===""")
print("""
Open TradingView for NVDA on 2025-10-29.

Check these values match:

1. ORB High (green line): ______ vs Python: {:.2f}
2. ORB Low (red line):    ______ vs Python: {:.2f}
3. First breakout bar time: ______ vs Python: {}

If they match → Phase 1 ORB detection is accurate ✓
If different → We need to debug
""".format(
    day_df['orb_high'].iloc[-1],
    day_df['orb_low'].iloc[-1],
    breakouts_high.index[0] if len(breakouts_high) > 0 else "No breakout"
))


=== VALIDATION CHECKLIST ===

Open TradingView for NVDA on 2025-10-29.

Check these values match:

1. ORB High (green line): ______ vs Python: 211.08
2. ORB Low (red line):    ______ vs Python: 207.09
3. First breakout bar time: ______ vs Python: 2025-10-29 09:59:00-04:00

If they match → Phase 1 ORB detection is accurate ✓
If different → We need to debug



## Next Steps

Once ORB detection matches:
- Phase 2: Stop loss calculation + R:R gate
- Phase 3: Confluence scoring (SSL, WAE, QQE)
- Phase 4: Exit management
- Phase 5: Adaptive systems